# Stage1 原始披露 HHI 单坐标信息检验

冻结原 Stage1 39 输入、H5 SOXX 开盘收益标签、2011–2022 训练和复用的 2023–2026 评价。候选只加先前已申报的 30 股披露市值 HHI，预期系数为负；0bp 完整账户是通过信息门后的独立步骤。

In [1]:
import os,json,hashlib,sys,sklearn
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd();card=root/'research/stage1_soxx_disclosure_hhi_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized'] and sys.version_info[:2]==(3,12) and sklearn.__version__=='1.9.1'
parent=plan['parent'];parent_card=root/parent['source_card']
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:assert sha(parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/name)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
spec=plan['concentration_sources'];assert sha(root/spec['train_summary'])==spec['train_summary_sha256'] and sha(root/spec['held_summary'])==spec['held_summary_sha256']
train_summary=json.loads((root/spec['train_summary']).read_text());held_summary=json.loads((root/spec['held_summary']).read_text())
assert train_summary['recorder_id']==spec['train_recorder_id'] and held_summary['recorder_id']==spec['held_recorder_id'] and train_summary['nas_readback']['verified'] and held_summary['nas_readback']['verified']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(train_summary['recorder_uri']):train_map=R.get_recorder(recorder_id=train_summary['recorder_id'],experiment_id=train_summary['experiment_id']).load_object('analysis/mapped_signals')
with R.uri_context(held_summary['recorder_uri']):held_map=R.get_recorder(recorder_id=held_summary['recorder_id'],experiment_id=held_summary['experiment_id']).load_object('analysis/mapped_signals')
assert len(train_map)==3020 and len(held_map)==944 and train_map.qualified_source.all() and held_map.source_known.all()
assert train_map.signal_date.is_unique and held_map.signal_date.is_unique and set(train_map.signal_date).isdisjoint(set(held_map.signal_date))
source=pd.concat([train_map[['signal_date','HHI']],held_map[['signal_date','HHI']]],ignore_index=True).sort_values('signal_date')
assert len(source)==3964 and source.signal_date.is_unique
hhi=pd.Series(source.HHI.to_numpy(float),index=pd.DatetimeIndex(source.signal_date),name=plan['single_mechanism']['feature']).reindex(frame.index)
forward=frame['soxx_forward_open_return'].astype(float);parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert train_dates.sum()==plan['cohorts']['expected_mature_train_rows'] and held_dates.sum()==plan['cohorts']['expected_mature_held_rows']
assert hhi.loc[train_dates|held_dates].notna().all() and np.isfinite(hhi.loc[train_dates|held_dates]).all()
augmented=base[base_names].copy();augmented[hhi.name]=hhi
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
train_prepared=dataset.prepare('train',col_set=['feature','label','meta']);held_prepared=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'Stage1_original_disclosure_HHI_DatasetH_label_maturity_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'parent_recorder_id':parent['recorder_id'],'train_source_recorder_id':train_summary['recorder_id'],'held_source_recorder_id':held_summary['recorder_id'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_calendar_rows':len(train_map),'held_calendar_rows':len(held_map),'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_unknown_rows':int(hhi.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(hhi.loc[held_dates].isna().sum()),'training_HHI_min':float(hhi.loc[train_dates].min()),'training_HHI_max':float(hhi.loc[train_dates].max()),'source_Recorder_object_loads':6,'DatasetH_builds':1,'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False}
receipt=card/'preflight_result.json'
if receipt.exists():assert json.loads(receipt.read_text())==report
else:receipt.write_text(json.dumps(report,indent=2)+'\n')
print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_unknown_rows','candidate_held_unknown_rows','training_HHI_min','training_HHI_max')})


2026/10/23 03:22:36 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[69044:MainThread](2026-10-23 03:22:36,418) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[69044:MainThread](2026-10-23 03:22:36,420) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[69044:MainThread](2026-10-23 03:22:36,420) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[69044:MainThread](2026-10-23 03:22:37,890) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[69044:MainThread](2026-10-23 03:22:37,890) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[69044:MainThread](2026-10-23 03:22:37,890) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'status': 'Stage1_original_disclosure_HHI_DatasetH_label_maturity_preflight_passed_no_fit', 'training_mature_rows': 3014, 'held_mature_rows': 940, 'candidate_train_unknown_rows': 0, 'candidate_held_unknown_rows': 0, 'training_HHI_min': 0.04516447605840671, 'training_HHI_max': 0.05405313693416049}


## 信息检验执行门

先完成 Qlib DatasetH、来源日期和标签成熟检查；固定两次 Ridge 拟合与一次区块检验只能在代码及计划检查后执行。

In [ ]:
import shlex,tarfile,tempfile,subprocess
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
review_path=card/'design_review_final.json';rehydration_path=card/'rehydration_plan.json'
assert review_path.exists() and rehydration_path.exists()
review=json.loads(review_path.read_text());rehydration=json.loads(rehydration_path.read_text())
notebook=json.loads((card/(card.name+'.ipynb')).read_text());fit_source=''.join(notebook['cells'][3]['source']);fit_hash=hashlib.sha256(fit_source.encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['preflight_sha256']==sha(card/'preflight_result.json') and review['rehydration_plan_sha256']==sha(rehydration_path) and review['fit_cell_source_sha256']==fit_hash
assert rehydration['frozen'] and rehydration['additional_source_recorder_loads_max']==6 and report==json.loads((card/'preflight_result.json').read_text())
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists()
state={'plan_sha256':sha(card/'plan.json'),'fit_cell_source_sha256':fit_hash,'fit_attempts':[],'fits_done':[],'prediction_attempts':[],'predictions_done':[],'bootstrap_attempted':False,'new_native_accounts':0}
state_path.write_text(json.dumps(state,indent=2)+'\n')
tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)];he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows']==3014 and len(he)==report['held_mature_rows']==940
yname='soxx_forward_five_open_log_return';feature_name=plan['single_mechanism']['feature']
ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float);assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
inputs={'baseline':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),'candidate':(tr['feature'][base_names+[feature_name]].to_numpy(float),he['feature'][base_names+[feature_name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).all() and np.isfinite(inputs['candidate'][1][:,-1]).all()
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));state_path.write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True)
 recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,derived_coordinate=hhi,source_hashes={'train':spec['train_summary_sha256'],'held':spec['held_summary_sha256'],'original39':parent['feature_artifact_sha256']})
 assert len(recorder.load_object('inputs/dataset').prepare('train',col_set=['feature','label','meta']))==report['train_segment_rows']
 for family in ('baseline','candidate'):
  assert family not in state['fit_attempts'] and len(state['fit_attempts'])<plan['budget']['model_fit_attempts_max']
  state['fit_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
  recorder.save_objects(artifact_path='models',**{family:model});models[family]=model
  state['fits_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  state['prediction_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  prediction=model.predict(inputs[family][1]);assert np.isfinite(prediction).all()
  saved=pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})
  recorder.save_objects(artifact_path='predictions',**{family:saved});predictions[family]=prediction
  state['predictions_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  recorder.log_metrics(completed_fits=len(state['fits_done']),completed_predictions=len(state['predictions_done']))
 assert len(state['fits_done'])==len(state['predictions_done'])==2
 dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
 daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2;daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2;daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
 assert daily.signal_date.is_monotonic_increasing and len(daily)==940
 halves=[daily.loc[daily.signal_date.lt(pd.Timestamp('2025-01-01'))],daily.loc[daily.signal_date.ge(pd.Timestamp('2025-01-01'))]];assert min(map(len,halves))>300
 bootstrap=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempted']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
 interval,_,replicates=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],alpha=bootstrap['one_sided_alpha'])
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
 metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),'candidate_held_mse':float(daily.candidate_squared_error.mean()),'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],'candidate_standardized_coordinate_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_negative':coefficient<0,'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
 passed=bool(all(gates.values()))
 daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str)
 quarters=[{'quarter':q,'sessions':len(group),'paired_mean_mse_improvement':float(group.paired_mse_improvement.mean())} for q,group in daily.groupby('quarter',sort=True)]
 recorder.save_objects(artifact_path='analysis/daily',held=daily)
 recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,bootstrap_replicates=replicates,information_gate_pass=passed)
 recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':'conditional_development_original_disclosure_HHI_information_pass_account_pending' if passed else 'not_promoted_fixed_original_disclosure_HHI_information_failed','objective_achieved':False,'information_gate_pass':passed,'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'reused_market_history':True,'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'parent_feature_sha256':parent['feature_artifact_sha256'],'source_train_summary_sha256':spec['train_summary_sha256'],'source_held_summary_sha256':spec['held_summary_sha256'],'artifact_policy':plan['artifact_policy'],'limitations':['Information MSE is not a fee-net SOXX funded account','Historical 2023-2026 is reused development feedback, not independent forward evidence','A later account must keep original Stage1 action/risk/recovery and pass all 0bp SOXX hard gates']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
(card/'result_review.json').write_text(json.dumps({'status':summary['decision'],'plan_sha256':summary['plan_sha256'],'recorder_id':state['recorder_id'],'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0},indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='soxx-hhi-info-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/models/baseline') for name in names) and any(name.endswith('/artifacts/models/candidate') for name in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['completed']=True;state['nas_archive_sha256']=evidence['sha256'];state_path.write_text(json.dumps(state,indent=2)+'\n')
print({k:summary[k] for k in ('decision','information_gate_pass','new_model_fits','new_native_accounts','recorder_id')});print(metrics)


## 中文结论

Qlib 无拟合预检确认 3,014/940 条成熟行均有此前已申报的 HHI，来源缺失 0。独立审阅 `design_review_final.json` 结论 HOLD_DUPLICATE：同源同 H5 目标的 `1−前五股占比` 卡已先冻结，且两卡都占用 `.025/27` 校正序号；HHI 基线缺少其共同来源年龄/表单/指数时期控制。此卡在拟合前关闭，0 拟合、0 区块检验、0 账户。保留预检与审阅，继续先冻结的单一 reported-diversity 路线。